# Sランクリスト — Colab 実行版（Phase 1 データ作成）

このノートブックは GitHub リポジトリの `src/` と **同一の処理ロジック**で S ランクリストを作成します。
違いは **認証方法だけ**です：

| | 認証 | 実行 |
|---|---|---|
| **GitHub Actions** | service account key (secret) | `python -m src.main` |
| **この Colab** | 実行中の Google アカウント自身の権限（service account 不要） | 上から順にセルを実行 |

**使い方:** メニューの `ランタイム → すべてのセルを実行` （Runtime → Run all）。
実行アカウントが次のすべてにアクセスできる必要があります:
- 入力: Drive フォルダ内の CSV 群
- 出力: BigQuery `vn-da-498509.SD`、および出力スプレッドシート

> 各セルは `src/` の各モジュールをそのまま inline したものです（ロジックは `src/main.py` の実行順と一致）。


In [ ]:
# @title 📚 ライブラリのインストール
# GitHub の requirements.txt と同等。Colab に既にある物も多いが、gspread 系は最新化しておく。
!pip install -q --upgrade gspread gspread-dataframe
!pip install -q "google-cloud-bigquery>=3.11.0" "google-api-python-client>=2.90.0" \
                "pyarrow>=14.0.0" "pandas-gbq>=0.26.1" "pandas>=2.0.0"
print("✓ ライブラリのインストール完了（必要なら『ランタイムを再起動』してから続行）")


In [ ]:
# @title 🔐 認証（Colab 実行アカウントの権限を使用）
# GitHub では service account key で認証するが、Colab では実行中の Google アカウントで認証する。
# → GCP_SERVICE_ACCOUNT_KEY は不要。アカウント自身が Drive / Sheets / BigQuery の権限を持っていること。
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
from google.cloud import bigquery
from googleapiclient.discovery import build

creds, _ = default()
drive_service = build("drive", "v3", credentials=creds)
gc = gspread.authorize(creds)
print("✓ 認証完了：drive_service（Drive API）と gc（gspread）を初期化しました")


In [ ]:
# @title ⚙️ 設定（定数）＆ 共通 import
# src/config.py に対応。PROJECT_ID / DATASET_ID は GitHub では secret だが、
# 実体は単なるリソース ID（秘匿情報ではない）なのでここに直書きする。
import io, re
import pandas as pd
from googleapiclient.http import MediaIoBaseDownload
from gspread_dataframe import set_with_dataframe

# ── BigQuery ──
PROJECT_ID = "vn-da-498509"
DATASET_ID = "SD"
bq_client = bigquery.Client(project=PROJECT_ID, credentials=creds)

BQ_TABLE_MR_MASTER = "mr_master"
BQ_TABLE_HOSPITAL_MASTER = "hospital_master"
BQ_TABLE_LP_LABEL_MASTER = "lp_label_master"

# ── Drive フォルダ ID（入力 CSV） ──
FOLDER_ID_MR_INFO = "14vPSh8Jqmf9N1iPmQKzar-W_crGr4-Px"
FOLDER_ID_HANDLING_HOSPITAL = "1Dfr-Pbax7CfBBBfBfVT-3HhhZvZwWoFI"
FOLDER_ID_HP_MASTER = "1WgUeoddIxwV_4qwscCcgor4CUPvpvvqS"
FOLDER_ID_LITEPLAN = "10GZeXE0AYTekK-A7Wv1kfMSLu4mwijFw"
FOLDER_ID_DR_ACTIVITY = "123W8yRftyjPNFK_7wcn87uCyEwRCAwmh"

# ── 出力スプレッドシート ID ──
SPREADSHEET_ID_MAIN = "1FYceOW233uL6fThahPHmaecJAhNKdwedHmrVocYsS_I"
SPREADSHEET_ID_OUT = "1oEIr4zbl8YPrbiPqXXL1KQ-DAD1JV4lGeuZw64O8pfg"

# ── シート名 ──
SHEET_MR_MASTER_INFO = "Mr Master Info"
SHEET_DR_USER_ACTIVE = "5.Dr user active"
SHEET_S_RANK = "Sランクリスト"
SHEET_LOG = "Log"
SHEET_ACTIVE_DR_USER = "acvite dr user"
SHEET_CASE1 = "List_dr_LP_hospital"
SHEET_CASE2 = "List_dr_without_LP_hospital"

print("✓ 設定を読み込み、bq_client を初期化しました")


In [ ]:
# @title 📂 Drive ヘルパー（src/drive_utils.py）
"""Helpers for reading CSV files out of Google Drive folders via the Drive API."""


def list_files_in_folder(drive_service, folder_id, name_pattern=None, exact_name=None):
    query = f"'{folder_id}' in parents and trashed=false"
    if exact_name:
        query += f" and name='{exact_name}'"

    results = drive_service.files().list(
        q=query,
        fields="files(id, name)",
        pageSize=100,
        supportsAllDrives=True,
        includeItemsFromAllDrives=True,
    ).execute()

    files = results.get("files", [])
    if name_pattern:
        files = [f for f in files if name_pattern.match(f["name"])]
    return sorted(files, key=lambda f: f["name"])


def read_csv_from_drive(drive_service, file_id, file_name) -> pd.DataFrame:
    request = drive_service.files().get_media(fileId=file_id)
    buffer = io.BytesIO()
    downloader = MediaIoBaseDownload(buffer, request)
    done = False
    while not done:
        _, done = downloader.next_chunk()
    buffer.seek(0)

    df = pd.read_csv(buffer)
    df["_source_file"] = file_name
    return df


def load_csv_by_name(drive_service, folder_id, filename) -> pd.DataFrame:
    """Find a single file by exact name in a folder and read it as a DataFrame."""
    files = list_files_in_folder(drive_service, folder_id, exact_name=filename)
    if not files:
        raise FileNotFoundError(f"'{filename}' not found in Drive folder '{folder_id}'")
    df = read_csv_from_drive(drive_service, files[0]["id"], files[0]["name"])
    return df.drop(columns=["_source_file", "Unnamed: 0"], errors="ignore")


def load_and_concat_csvs(drive_service, folder_id, name_pattern) -> pd.DataFrame:
    """Find all files matching a regex pattern in a folder and concat them into one DataFrame."""
    files = list_files_in_folder(drive_service, folder_id, name_pattern=name_pattern)
    print(f"✓ Found {len(files)} matching file(s) in folder {folder_id}")

    dfs = []
    for f in files:
        try:
            dfs.append(read_csv_from_drive(drive_service, f["id"], f["name"]))
        except Exception as e:
            print(f"⚠ Failed to read {f['name']}: {e}")

    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()


In [ ]:
# @title 📝 Sheets ヘルパー（src/sheet_utils.py）
"""Shared Google Sheets write patterns used throughout the pipeline.

Pipeline A always does clear + full overwrite for full-table sheets (idempotent
per run) and uses R{row}C{col}-ranged updates for single-column write-backs,
which assumes row order matches the existing sheet.
"""


def _drop_spurious_columns(df):
    """Remove auto-generated extra columns such as 'Column 12' or 'Unnamed: 0'."""
    spurious_columns = [
        col
        for col in df.columns
        if isinstance(col, str)
        and (
            col.startswith("Unnamed:") or re.fullmatch(r"Column\s+\d+", col)
        )
    ]
    return df.drop(columns=spurious_columns, errors="ignore")


def get_or_create_worksheet(spreadsheet, title: str):
    try:
        return spreadsheet.worksheet(title)
    except gspread.exceptions.WorksheetNotFound:
        return spreadsheet.add_worksheet(title=title, rows=1, cols=1)


def overwrite_sheet_with_dataframe(worksheet, df, cast_to_str: bool = True):
    """Clear a sheet and write the full DataFrame back.

    cast_to_str=True (the default, used for every derived/output table) matches the
    source notebook's `.fillna("").astype(str)` pattern. The two raw-CSV mirror
    writes (menkai/chat status -> their tracking sheets) only do `.fillna("")` in
    the original, so pass cast_to_str=False there to match.
    """
    worksheet.clear()
    df = _drop_spurious_columns(df)
    df = df.fillna("")
    if cast_to_str:
        df = df.astype(str)

    df = df.drop(columns=["Column 12", "Column 13"], errors="ignore")
    set_with_dataframe(worksheet, df)


def write_column(worksheet, column_name: str, values: list) -> int:
    """Write a single column back to a sheet, creating the header if needed."""
    headers = worksheet.row_values(1)
    if column_name in headers:
        col_idx = headers.index(column_name) + 1
    else:
        col_idx = len(headers) + 1
        worksheet.update_cell(1, col_idx, column_name)

    if not values:
        # Nothing to write - R2C{col}:R1C{col} would be an inverted (invalid) range.
        return col_idx

    worksheet.update(
        [[v] for v in values],
        f"R2C{col_idx}:R{len(values) + 1}C{col_idx}",
    )
    return col_idx


In [ ]:
# @title 1️⃣ MR master（Steps 1–12）
"""Phase 1, Steps 1-12: build & push `mr_master` (Pr.JOY MR users + assigned hospital count)."""


FILE_PATTERN_MR_INFO = re.compile(r"^pr_registration_officeusers_\d{4}\.csv$")
FILE_PATTERN_HANDLING_HOSPITAL = re.compile(r"^handlingHospital_\d{4}\.csv$")

MR_INFO_COLUMNS = [
    "officeUserId", "created", "userName",
    "age", "sex", "jobName",
    "officeId", "officeName", "officeType",
]


def build_mr_master(drive_service):
    df_prjoy_userinfo = load_and_concat_csvs(drive_service, FOLDER_ID_MR_INFO, FILE_PATTERN_MR_INFO)
    print(f"✓ df_prjoy_userinfo: {df_prjoy_userinfo.shape[0]:,} rows x {df_prjoy_userinfo.shape[1]} cols")

    df_prjoy_userinfo = df_prjoy_userinfo[MR_INFO_COLUMNS]

    handling_hp = load_and_concat_csvs(
        drive_service, FOLDER_ID_HANDLING_HOSPITAL, FILE_PATTERN_HANDLING_HOSPITAL
    )
    print(f"✓ handling_hp: {handling_hp.shape[0]:,} rows x {handling_hp.shape[1]} cols")

    mr_hospital_count = (
        handling_hp
        .groupby("officeuserid")["officeid"]
        .nunique()
        .reset_index()
        .rename(columns={"officeid": "assigned_hospital_count"})
    )

    df_prjoy_userinfo = df_prjoy_userinfo.merge(
        mr_hospital_count,
        left_on="officeUserId",
        right_on="officeuserid",
        how="left",
    ).drop(columns=["officeuserid"])

    df_prjoy_userinfo = df_prjoy_userinfo[df_prjoy_userinfo["assigned_hospital_count"].notna()]
    print(f"✓ df_prjoy_userinfo after filtering: {df_prjoy_userinfo.shape[0]:,} rows x {df_prjoy_userinfo.shape[1]} cols")

    return df_prjoy_userinfo, handling_hp


def push_mr_master_to_bigquery(bq_client, df_prjoy_userinfo):
    table_id = f"{PROJECT_ID}.{DATASET_ID}.{BQ_TABLE_MR_MASTER}"
    job = bq_client.load_table_from_dataframe(
        df_prjoy_userinfo,
        table_id,
        job_config=bigquery.LoadJobConfig(write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE),
    )
    job.result()
    print(f"✓ Pushed {df_prjoy_userinfo.shape[0]:,} rows to {table_id}")


def prepare_mr_master_info_sheet(gspread_client, df_prjoy_userinfo):
    """Step 12: clears the Mr Master Info sheet. The actual worksheet.update() write is
    intentionally left disabled here, matching the original notebook's behavior."""
    spreadsheet = gspread_client.open_by_key(SPREADSHEET_ID_MAIN)
    worksheet = get_or_create_worksheet(spreadsheet, SHEET_MR_MASTER_INFO)
    worksheet.clear()

    # data = [df_prjoy_userinfo.columns.tolist()] + df_prjoy_userinfo.fillna("").astype(str).values.tolist()
    # worksheet.update(data, value_input_option="RAW")  # intentionally disabled, see README

    print(f"✓ Cleared sheet '{SHEET_MR_MASTER_INFO}' (write disabled, matches source notebook)")


In [ ]:
# @title 2️⃣ Hospital master（Steps 13–18）
"""Phase 1, Steps 13-18: build & push `hospital_master` (MR<->hospital assignments)."""


def build_hospital_master(drive_service, handling_hp):
    handling_hp_final = (
        handling_hp[["officeuserid", "officeid", "start"]]
        .rename(columns={
            "officeuserid": "officeUserId",
            "officeid": "officeId",
            "start": "assigned_start_date",
        })
    )
    print(f"✓ handling_hp_final: {handling_hp_final.shape[0]:,} rows x {handling_hp_final.shape[1]} cols")

    hp_master_files = list_files_in_folder(
        drive_service, FOLDER_ID_HP_MASTER, exact_name="hp_master_df.csv"
    )
    if not hp_master_files:
        raise FileNotFoundError("hp_master_df.csv not found in Drive folder")

    hp_master_df = read_csv_from_drive(drive_service, hp_master_files[0]["id"], hp_master_files[0]["name"])
    hp_master_df = (
        hp_master_df[["officeId", "都道府県", "全担当者数"]]
        .rename(columns={"都道府県": "prefecture", "全担当者数": "assigned_pic_count"})
    )

    handling_hp_final = handling_hp_final.merge(hp_master_df, on="officeId", how="left")
    print(f"✓ handling_hp_final after join: {handling_hp_final.shape[0]:,} rows x {handling_hp_final.shape[1]} cols")

    return handling_hp_final


def push_hospital_master_to_bigquery(bq_client, handling_hp_final):
    table_id = f"{PROJECT_ID}.{DATASET_ID}.{BQ_TABLE_HOSPITAL_MASTER}"
    job = bq_client.load_table_from_dataframe(
        handling_hp_final,
        table_id,
        job_config=bigquery.LoadJobConfig(
            write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
            autodetect=True,
        ),
    )
    job.result()
    print(f"✓ Pushed {handling_hp_final.shape[0]:,} rows to {table_id}")


In [ ]:
# @title 3️⃣ Lite Plan label（Steps 19–23）
"""Phase 1, Steps 19-23: build & push `lp_label_master` (Lite Plan purchases)."""


def build_lp_info(drive_service):
    files = list_files_in_folder(drive_service, FOLDER_ID_LITEPLAN, exact_name="liteplan_payment.csv")
    if not files:
        raise FileNotFoundError("liteplan_payment.csv not found in Drive folder")

    lp_info = read_csv_from_drive(drive_service, files[0]["id"], files[0]["name"])
    print(f"✓ lp_info raw: {lp_info.shape[0]:,} rows x {lp_info.shape[1]} cols")

    # drop rows where both cancellation-date columns are non-null (fully cancelled)
    lp_info = lp_info[lp_info["解除予約日"].isna() | lp_info["解除完了日"].isna()]

    lp_info = lp_info.assign(
        key=lp_info["ユーザーID"].astype(str) + "_" + lp_info["ライト購入施設ID"].astype(str)
    )
    lp_info = lp_info.assign(購入時刻=pd.to_datetime(lp_info["購入時刻"], errors="coerce"))

    today = pd.Timestamp.now().normalize()
    one_month_ago = today - pd.DateOffset(months=1)
    lp_info = lp_info.assign(
        is_first_purchase=lp_info["購入時刻"].between(one_month_ago, today)
    )
    lp_info = lp_info.assign(
        number_of_contract=lp_info["ユーザーID"].map(lp_info["ユーザーID"].value_counts())
    )

    lp_info = (
        lp_info[["ユーザーID", "氏名", "ライト購入施設ID", "購入時刻", "is_first_purchase", "number_of_contract"]]
        .rename(columns={
            "ユーザーID": "officeuserid",
            "氏名": "mr_name",
            "ライト購入施設ID": "officeid",
            "購入時刻": "purchase_date",
        })
    )
    print(f"✓ lp_info final: {lp_info.shape[0]:,} rows x {lp_info.shape[1]} cols")

    return lp_info


def push_lp_label_master_to_bigquery(bq_client, lp_info):
    table_id = f"{PROJECT_ID}.{DATASET_ID}.{BQ_TABLE_LP_LABEL_MASTER}"
    job = bq_client.load_table_from_dataframe(
        lp_info,
        table_id,
        job_config=bigquery.LoadJobConfig(
            write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
            autodetect=True,
        ),
    )
    job.result()
    print(f"✓ Pushed {lp_info.shape[0]:,} rows to {table_id}")


In [ ]:
# @title 4️⃣ Dr active / menkai / chat（Steps 24–26）
"""Phase 1, Steps 24-26: load Dr active/menkai/chat status, write back meeting request date."""


def load_dr_activity_data(drive_service):
    df_dractive_users = load_csv_by_name(drive_service, FOLDER_ID_DR_ACTIVITY, "Dr_active_user.csv")
    df_dractive_users["オフィスユーザーID"] = df_dractive_users["オフィスユーザーID"].astype(str).str.strip()

    df_menkai = load_csv_by_name(drive_service, FOLDER_ID_DR_ACTIVITY, "dr_menkai_status.csv")
    df_menkai["Dr Office User ID"] = df_menkai["Dr Office User ID"].astype(str).str.strip()
    df_menkai = df_menkai[df_menkai["面会ステータス"].astype(str).str.strip() == "FIXED"]
    print(f"✓ df_menkai filtered to 面会ステータス=='FIXED': {df_menkai.shape[0]:,} rows")

    df_dr_chat = load_csv_by_name(drive_service, FOLDER_ID_DR_ACTIVITY, "dr_chat_status.csv")
    df_dr_chat["送信者ID"] = df_dr_chat["送信者ID"].astype(str).str.strip()

    print(f"✓ df_dractive_users: {df_dractive_users.shape[0]:,} rows x {df_dractive_users.shape[1]} cols")
    print(f"✓ df_menkai        : {df_menkai.shape[0]:,} rows x {df_menkai.shape[1]} cols")
    print(f"✓ df_dr_chat       : {df_dr_chat.shape[0]:,} rows x {df_dr_chat.shape[1]} cols")

    return df_dractive_users, df_menkai, df_dr_chat


def enrich_dr_active_users(df_dractive_users, df_menkai, df_dr_chat):
    """Merge latest chat 'Updated Date' and latest 面会リクエスト日 (meeting request date) per Dr."""
    if "Updated Date" in df_dractive_users.columns:
        df_dractive_users = df_dractive_users.drop(columns=["Updated Date"])

    chat_map_dr = (
        df_dr_chat[["送信者ID", "Updated Date"]]
        .groupby("送信者ID", as_index=False)["Updated Date"]
        .max()
        .rename(columns={"送信者ID": "オフィスユーザーID"})
    )
    df_dractive_users = df_dractive_users.merge(chat_map_dr, on="オフィスユーザーID", how="left")

    if "面会リクエスト日" in df_dractive_users.columns:
        df_dractive_users = df_dractive_users.drop(columns=["面会リクエスト日"])

    menkai_map = (
        df_menkai[["Dr Office User ID", "リクエスト日時 Date"]]
        .groupby("Dr Office User ID", as_index=False)["リクエスト日時 Date"]
        .max()
        .rename(columns={
            "Dr Office User ID": "オフィスユーザーID",
            "リクエスト日時 Date": "面会リクエスト日",
        })
    )
    df_dractive_users = df_dractive_users.merge(menkai_map, on="オフィスユーザーID", how="left")

    print(f"✓ df_dractive_users enriched: {df_dractive_users.shape[0]:,} rows x {df_dractive_users.shape[1]} cols")
    return df_dractive_users


def write_back_meeting_request_date(spreadsheet_main, df_dractive_users):
    ws_dractive = spreadsheet_main.worksheet(SHEET_DR_USER_ACTIVE)
    values = df_dractive_users["面会リクエスト日"].fillna("").tolist()
    col_idx = write_column(ws_dractive, "面会リクエスト日", values)
    print(f"✓ Wrote '面会リクエスト日' -> column {col_idx} of sheet '{SHEET_DR_USER_ACTIVE}' ({len(values):,} rows)")


In [ ]:
# @title 5️⃣ S-rank list（Steps 27–36）
"""Phase 1, Steps 27-36: filter new Lite Plan buyers, join menkai/chat status, build & push s_rank_df."""


def filter_new_lite_plan_purchases(lp_info, df_prjoy_userinfo):
    """Step 27-29: brand-new, single-contract Lite Plan buyers with >=2 assigned
    hospitals, purchased within the last 15 days."""
    lp_info_new = lp_info[
        (lp_info["is_first_purchase"] == True) & (lp_info["number_of_contract"] == 1)
    ].copy()
    print(f"✓ lp_info_new after is_first_purchase/number_of_contract filter: {lp_info_new.shape[0]:,} rows")

    lp_info_new = lp_info_new.merge(
        df_prjoy_userinfo[["officeUserId", "assigned_hospital_count"]],
        left_on="officeuserid",
        right_on="officeUserId",
        how="left",
    ).drop(columns=["officeUserId"])

    lp_info_new = lp_info_new[lp_info_new["assigned_hospital_count"] >= 2].copy()

    lp_info_new = lp_info_new.assign(
        purchase_date=pd.to_datetime(lp_info_new["purchase_date"], errors="coerce").dt.normalize()
    )
    today = pd.Timestamp.now().normalize()
    lp_info_new = lp_info_new[(today - lp_info_new["purchase_date"]).dt.days <= 15]
    print(f"✓ lp_info_new after assigned_hospital_count/purchase_date filter: {lp_info_new.shape[0]:,} rows")

    return lp_info_new


def join_menkai_and_chat(drive_service, lp_info_new):
    """Step 30-33: join latest MR menkai request date and aggregated MR chat status.

    The raw menkai/chat CSVs are only used in-memory here; they are no longer
    mirrored to tracking sheets (data outgrew the Sheets cell limit).
    """
    df_mr_menkai_raw = load_csv_by_name(drive_service, FOLDER_ID_DR_ACTIVITY, "dr_menkai_status.csv")

    df_mr_menkai = df_mr_menkai_raw.copy()
    df_mr_menkai["Mr Office User ID"] = df_mr_menkai["Mr Office User ID"].astype(str).str.strip()
    df_mr_menkai["面会ステータス"] = df_mr_menkai["面会ステータス"].astype(str).str.strip()

    # Total FIXED / NEW menkai request counts per MR (from all rows, before the NEW-only
    # filter below). These two columns ride along to the Sランクリスト output sheet.
    fixed_count = (
        df_mr_menkai[df_mr_menkai["面会ステータス"] == "FIXED"]
        .groupby("Mr Office User ID").size().rename("menkai_fixed_count")
    )
    new_count = (
        df_mr_menkai[df_mr_menkai["面会ステータス"] == "NEW"]
        .groupby("Mr Office User ID").size().rename("menkai_new_count")
    )
    menkai_counts = (
        pd.concat([fixed_count, new_count], axis=1)
        .fillna(0).astype(int)
        .reset_index()
        .rename(columns={"Mr Office User ID": "officeuserid"})
    )

    df_mr_menkai = df_mr_menkai[df_mr_menkai["面会ステータス"] == "NEW"]

    if "リクエスト日時 Date" in lp_info_new.columns:
        lp_info_new = lp_info_new.drop(columns=["リクエスト日時 Date"])

    menkai_mr_map = (
        df_mr_menkai[["Mr Office User ID", "リクエスト日時 Date"]]
        .rename(columns={"Mr Office User ID": "officeuserid"})
        .groupby("officeuserid", as_index=False)["リクエスト日時 Date"]
        .max()
    )
    lp_info_new = lp_info_new.merge(menkai_mr_map, on="officeuserid", how="left")

    for col in ["menkai_fixed_count", "menkai_new_count"]:
        if col in lp_info_new.columns:
            lp_info_new = lp_info_new.drop(columns=[col])
    lp_info_new = lp_info_new.merge(menkai_counts, on="officeuserid", how="left")
    lp_info_new["menkai_fixed_count"] = lp_info_new["menkai_fixed_count"].fillna(0).astype(int)
    lp_info_new["menkai_new_count"] = lp_info_new["menkai_new_count"].fillna(0).astype(int)

    df_mr_chat_raw = load_csv_by_name(drive_service, FOLDER_ID_DR_ACTIVITY, "mr_chat_status.csv")
    df_mr_chat = df_mr_chat_raw.copy()
    df_mr_chat["送信者ID"] = df_mr_chat["送信者ID"].astype(str).str.strip()

    chat_cols = ["送信者ID", "未読のメッセージ数", "既読者ID", "Updated Date"]
    for col in ["未読のメッセージ数", "既読者ID", "Updated Date"]:
        if col in lp_info_new.columns:
            lp_info_new = lp_info_new.drop(columns=[col])

    chat_map = (
        df_mr_chat[chat_cols]
        .rename(columns={"送信者ID": "officeuserid"})
        .groupby("officeuserid", as_index=False)
        .agg({"未読のメッセージ数": "sum", "既読者ID": "last", "Updated Date": "max"})
    )
    lp_info_new = lp_info_new.merge(chat_map, on="officeuserid", how="left")
    print(f"✓ lp_info_new after menkai/chat join: {lp_info_new.shape[0]:,} rows x {lp_info_new.shape[1]} cols")

    return lp_info_new


def build_s_rank_df(lp_info_new):
    """Step 35: rows with a stuck (unread) chat >=5 days old OR a stuck menkai request >=5 days old."""
    today = pd.Timestamp.now().normalize()
    updated_date = pd.to_datetime(lp_info_new["Updated Date"], errors="coerce").dt.normalize()
    request_date = pd.to_datetime(lp_info_new["リクエスト日時 Date"], errors="coerce").dt.normalize()

    cond1 = (
        ((today - updated_date).dt.days >= 5)
        & (lp_info_new["既読者ID"].isna() | (lp_info_new["既読者ID"].astype(str).str.strip() == ""))
    )
    cond2 = (today - request_date).dt.days >= 5

    mask = cond1 | cond2
    s_rank_df = lp_info_new[mask].copy()

    # S判定条件: which condition(s) qualified this row for rank S -
    # cond1 (stuck unread chat) -> "メッセージ", cond2 (stuck menkai request) -> "面会".
    def _judge(has_chat, has_menkai):
        labels = []
        if has_chat:
            labels.append("メッセージ")
        if has_menkai:
            labels.append("面会")
        return "・".join(labels)

    s_rank_df["S判定条件"] = [
        _judge(bool(a), bool(b)) for a, b in zip(cond1[mask], cond2[mask])
    ]
    print(f"✓ s_rank_df: {s_rank_df.shape[0]:,} rows (cond1={cond1.sum():,}, cond2={cond2.sum():,})")

    return s_rank_df


SENT_WINDOW_DAYS = 30


def office_ids_sent_recently(spreadsheet_out):
    """Office user IDs already logged (sent) within the last SENT_WINDOW_DAYS days, from the 'Log' sheet.

    The Log sheet mirrors the output rows plus a 送信日時 timestamp written when each
    row is appended. Anyone whose 送信日時 is within the last 30 days (up to now) has
    already been contacted recently and must not be sent again, so we exclude them
    s_rank_df uses 'officeuserid'; we match it case-insensitively. Fails safe
    (returns an empty set) if the Log sheet or its expected columns are absent.
    """
    try:
        ws_log = spreadsheet_out.worksheet(SHEET_LOG)
    except gspread.exceptions.WorksheetNotFound:
        print(f"⚠ Log sheet '{SHEET_LOG}' not found - skipping already-sent filter")
        return set()

    records = ws_log.get_all_records()
    if not records:
        return set()

    log_df = pd.DataFrame(records)
    id_col = next((c for c in log_df.columns if str(c).strip().lower() == "officeuserid"), None)
    if "送信日時" not in log_df.columns or id_col is None:
        print("⚠ Log sheet missing 'officeUserId' or '送信日時' - skipping already-sent filter")
        return set()

    # format="mixed" parses each 送信日時 value independently, so a stray date-only or
    # differently-formatted cell doesn't silently become NaT and slip through the filter.
    sent = pd.to_datetime(log_df["送信日時"], errors="coerce", format="mixed")
    today = pd.Timestamp.now().normalize()
    days_since = (today - sent.dt.normalize()).dt.days
    recent = days_since.between(0, SENT_WINDOW_DAYS)  # sent within the last 30 days (future dates excluded)
    ids = log_df.loc[recent, id_col].astype(str).str.strip()
    ids = set(ids[ids != ""])
    print(f"✓ {len(ids):,} officeuserid(s) sent within the last {SENT_WINDOW_DAYS} days (from '{SHEET_LOG}')")
    return ids


def push_s_rank_to_sheet(gspread_client, s_rank_df):
    """Step 36: push s_rank_df to the 'Sランクリスト' output sheet.

    Before writing, drop any officeuserid that was already sent within the last 30
    days (present in the 'Log' sheet with a 送信日時 in that window) so nobody is
    contacted twice inside a 30-day window.
    """
    spreadsheet_out = gspread_client.open_by_key(SPREADSHEET_ID_OUT)
    ws_out = get_or_create_worksheet(spreadsheet_out, SHEET_S_RANK)

    already_sent = office_ids_sent_recently(spreadsheet_out)
    if already_sent:
        before = len(s_rank_df)
        s_rank_df = s_rank_df[
            ~s_rank_df["officeuserid"].astype(str).str.strip().isin(already_sent)
        ].copy()
        print(f"✓ Excluded {before - len(s_rank_df):,} row(s) sent within the last {SENT_WINDOW_DAYS} days; {len(s_rank_df):,} remain")

    s_rank_df = s_rank_df.drop(
        columns=["is_first_purchase", "number_of_contract", "assigned_hospital_count"],
        errors="ignore",
    )
    s_rank_df = s_rank_df.rename(
        columns={
            "リクエスト日時 Date": "Request meeting time",
            "未読のメッセージ数": "Unread message",
            "既読者ID": "reader_id",
        },
        errors="ignore",
    )

    # Keep S判定条件 as the last data column so it sits directly before the
    # Pattern column that write_back_pattern_and_suggestion appends afterwards.
    if "S判定条件" in s_rank_df.columns:
        ordered = [c for c in s_rank_df.columns if c != "S判定条件"] + ["S判定条件"]
        s_rank_df = s_rank_df[ordered]

    overwrite_sheet_with_dataframe(ws_out, s_rank_df)
    print(f"✓ Pushed {s_rank_df.shape[0]:,} rows to sheet '{SHEET_S_RANK}'")

    return spreadsheet_out, s_rank_df

In [ ]:
# @title 6️⃣ Active Dr user（Steps 37–39）
"""Phase 1, Steps 37-39: build & push the active Dr user list."""


def filter_active_dr_users(df_dractive_users):
    df = df_dractive_users.rename(columns={
        "オフィスユーザーID": "officeUserId",
        "ユーザー名": "name",
        "企業ID": "officeId",
        "事業所名": "hospitalName",
        "最終利用時刻": "last_access_date",
        "Updated Date": "last_message_date",
        "面会リクエスト日": "last_meeting_date",
    })

    today = pd.Timestamp.now().normalize()
    seven_days_ago = today - pd.DateOffset(days=7)

    last_access = pd.to_datetime(df["last_access_date"], errors="coerce").dt.normalize()
    last_msg = pd.to_datetime(df["last_message_date"], errors="coerce")
    last_meet = pd.to_datetime(df["last_meeting_date"], errors="coerce")

    df = df[
        (last_access >= seven_days_ago)
        & (last_msg.notna() | last_meet.notna())
        & (~df["hospitalName"].astype(str).str.contains(r"Dr\.JOY|テスト", na=False))
        & (~df["name"].astype(str).str.contains("test", case=False, na=False))
    ].copy()

    print(f"✓ df_dractive_users_filtered: {df.shape[0]:,} rows x {df.shape[1]} cols")
    return df


def push_active_dr_users_to_sheet(spreadsheet_out, df_dractive_users_filtered):
    ws_active = get_or_create_worksheet(spreadsheet_out, SHEET_ACTIVE_DR_USER)
    overwrite_sheet_with_dataframe(ws_active, df_dractive_users_filtered)
    print(f"✓ Pushed {df_dractive_users_filtered.shape[0]:,} rows to sheet '{SHEET_ACTIVE_DR_USER}'")


In [ ]:
# @title 7️⃣ Case 1 / Case 2 + Pattern（Steps 40–46）
"""Phase 1, Steps 40-46: Case 1 / Case 2 opportunity detection, Pattern + suggested hospital name."""


def build_case1_active_dr_at_lp_hospital(s_rank_df, df_dractive_users_filtered):
    """Step 40: active Drs at the hospital the S-rank MR just got a Lite Plan purchase at."""
    s_rank_officeids = set(s_rank_df["officeid"].astype(str).unique())

    df_active_dr_at_mr_hp = df_dractive_users_filtered[
        df_dractive_users_filtered["officeId"].astype(str).isin(s_rank_officeids)
    ].copy()

    print(f"✓ df_active_dr_at_mr_hp: {df_active_dr_at_mr_hp.shape[0]:,} rows, "
          f"{df_active_dr_at_mr_hp['officeId'].nunique():,} unique hospitals")
    return df_active_dr_at_mr_hp


def push_case1_to_sheet(spreadsheet_out, df_active_dr_at_mr_hp):
    """Step 41."""
    ws_case1 = get_or_create_worksheet(spreadsheet_out, SHEET_CASE1)
    overwrite_sheet_with_dataframe(ws_case1, df_active_dr_at_mr_hp)
    print(f"✓ Pushed {df_active_dr_at_mr_hp.shape[0]:,} rows to sheet '{SHEET_CASE1}'")


def build_case2_hospitals_without_lp(s_rank_df, handling_hp_final, df_dractive_users):
    """Step 42-43: hospitals an S-rank MR covers but that are NOT the Lite-Plan hospital
    in s_rank_df, joined against active Drs at each such hospital."""
    s_rank_userids = set(s_rank_df["officeuserid"].astype(str).unique())
    s_rank_pairs = set(zip(s_rank_df["officeuserid"].astype(str), s_rank_df["officeid"].astype(str)))

    df_case2 = handling_hp_final[
        handling_hp_final["officeUserId"].astype(str).isin(s_rank_userids)
        & ~handling_hp_final.apply(
            lambda r: (str(r["officeUserId"]), str(r["officeId"])) in s_rank_pairs, axis=1
        )
    ].copy()
    print(f"✓ df_case2: {df_case2.shape[0]:,} rows, "
          f"{df_case2['officeUserId'].nunique():,} unique MRs, {df_case2['officeId'].nunique():,} unique hospitals")

    df_case2 = df_case2.rename(columns={"officeUserId": "MrOfficeUserId"})

    dr_for_case2 = df_dractive_users.rename(columns={
        "企業ID": "officeId",
        "オフィスユーザーID": "dr_officeUserId",
    })[["officeId", "dr_officeUserId"]].copy()

    dr_for_case2["officeId"] = dr_for_case2["officeId"].astype(str).str.strip()
    df_case2["officeId"] = df_case2["officeId"].astype(str).str.strip()
    df_case2["MrOfficeUserId"] = df_case2["MrOfficeUserId"].astype(str).str.strip()

    df_case2 = df_case2.merge(dr_for_case2, on="officeId", how="inner")

    hp_name_map = (
        df_dractive_users[["企業ID", "事業所名"]]
        .rename(columns={"企業ID": "officeId", "事業所名": "officeName"})
        .drop_duplicates(subset=["officeId"])
    )
    hp_name_map["officeId"] = hp_name_map["officeId"].astype(str).str.strip()

    df_case2 = df_case2.merge(hp_name_map, on="officeId", how="left")
    print(f"✓ df_case2 after Dr join: {df_case2.shape[0]:,} rows, "
          f"{df_case2['dr_officeUserId'].notna().sum():,} matched Drs")

    return df_case2


def push_case2_to_sheet(spreadsheet_out, df_case2):
    """Step 44."""
    ws_case2 = get_or_create_worksheet(spreadsheet_out, SHEET_CASE2)
    overwrite_sheet_with_dataframe(ws_case2, df_case2)
    print(f"✓ Pushed {df_case2.shape[0]:,} rows to sheet '{SHEET_CASE2}'")


def assign_pattern(s_rank_df, df_active_dr_at_mr_hp, df_case2):
    """Step 45: Pattern Sα-1 = Case1 only, Pattern Sα-2 = Case2 only, Pattern Sα-3 = both, Pattern 0 = neither."""
    case1_officeids = set(df_active_dr_at_mr_hp["officeId"].astype(str).unique())
    case2_mrids = set(df_case2["MrOfficeUserId"].astype(str).unique())

    def _pattern(row):
        in_case1 = str(row["officeid"]) in case1_officeids
        in_case2 = str(row["officeuserid"]) in case2_mrids
        if in_case1 and in_case2:
            return "Pattern Sα-3"
        elif in_case1:
            return "Pattern Sα-1"
        elif in_case2:
            return "Pattern Sα-2"
        return "Pattern 0"

    s_rank_df = s_rank_df.copy()
    s_rank_df["Pattern"] = s_rank_df.apply(_pattern, axis=1)
    print("✓ Pattern distribution:")
    print(s_rank_df["Pattern"].value_counts(dropna=False).to_string())

    return s_rank_df


def assign_suggested_hospital_name(s_rank_df, df_case2, max_hospitals=3):
    """Step 45b: for Pattern Sα-2/Sα-3 rows, list up to `max_hospitals` Case2 hospitals for that MR,
    ordered by Dr count desc."""
    suggest_map = (
        df_case2[["MrOfficeUserId", "officeName"]]
        .dropna(subset=["officeName"])
        .groupby("MrOfficeUserId")["officeName"]
        .apply(lambda x: ", ".join(
            name for name, _ in sorted(x.value_counts().items(), key=lambda t: t[1], reverse=True)[:max_hospitals]
        ))
        .reset_index()
        .rename(columns={"MrOfficeUserId": "officeuserid", "officeName": "suggest_hospital_name"})
    )

    s_rank_df = s_rank_df.copy()
    if "suggest_hospital_name" in s_rank_df.columns:
        s_rank_df = s_rank_df.drop(columns=["suggest_hospital_name"])

    s_rank_df = s_rank_df.merge(suggest_map, on="officeuserid", how="left")
    s_rank_df.loc[~s_rank_df["Pattern"].isin(["Pattern Sα-2", "Pattern Sα-3"]), "suggest_hospital_name"] = ""

    return s_rank_df


def write_back_pattern_and_suggestion(spreadsheet_out, s_rank_df):
    """Step 46: write both Pattern and suggest_hospital_name columns back to the sheet."""
    ws_out = spreadsheet_out.worksheet(SHEET_S_RANK)
    for column_name in ["Pattern", "suggest_hospital_name"]:
        values = s_rank_df[column_name].fillna("").tolist()
        col_idx = write_column(ws_out, column_name, values)
        print(f"✓ Wrote '{column_name}' -> column {col_idx} of sheet '{SHEET_S_RANK}' ({len(values):,} rows)")


In [ ]:
# @title ▶️ パイプライン実行（src/main.py と同じ順序）
def main():
    # Steps 1-12: MR master
    df_prjoy_userinfo, handling_hp = build_mr_master(drive_service)
    push_mr_master_to_bigquery(bq_client, df_prjoy_userinfo)
    prepare_mr_master_info_sheet(gc, df_prjoy_userinfo)

    # Steps 13-18: hospital master
    handling_hp_final = build_hospital_master(drive_service, handling_hp)
    push_hospital_master_to_bigquery(bq_client, handling_hp_final)

    # Steps 19-23: lp label master
    lp_info = build_lp_info(drive_service)
    push_lp_label_master_to_bigquery(bq_client, lp_info)

    # Steps 24-26: Dr activity + write back meeting request date
    spreadsheet_main = gc.open_by_key(SPREADSHEET_ID_MAIN)
    df_dractive_users, df_menkai, df_dr_chat = load_dr_activity_data(drive_service)
    df_dractive_users = enrich_dr_active_users(df_dractive_users, df_menkai, df_dr_chat)
    write_back_meeting_request_date(spreadsheet_main, df_dractive_users)

    # Steps 27-36: S-rank list
    lp_info_new = filter_new_lite_plan_purchases(lp_info, df_prjoy_userinfo)
    lp_info_new = join_menkai_and_chat(drive_service, lp_info_new)
    s_rank_df = build_s_rank_df(lp_info_new)
    spreadsheet_out, s_rank_df = push_s_rank_to_sheet(gc, s_rank_df)

    # Steps 37-39: active Dr users
    df_dractive_users_filtered = filter_active_dr_users(df_dractive_users)
    push_active_dr_users_to_sheet(spreadsheet_out, df_dractive_users_filtered)

    # Steps 40-44: Case 1 / Case 2 opportunity detection
    df_active_dr_at_mr_hp = build_case1_active_dr_at_lp_hospital(s_rank_df, df_dractive_users_filtered)
    push_case1_to_sheet(spreadsheet_out, df_active_dr_at_mr_hp)
    df_case2 = build_case2_hospitals_without_lp(s_rank_df, handling_hp_final, df_dractive_users)
    push_case2_to_sheet(spreadsheet_out, df_case2)

    # Steps 45-46: Pattern + suggested hospital name, write back
    s_rank_df = assign_pattern(s_rank_df, df_active_dr_at_mr_hp, df_case2)
    s_rank_df = assign_suggested_hospital_name(s_rank_df, df_case2)
    write_back_pattern_and_suggestion(spreadsheet_out, s_rank_df)

    print("✓ Phase 1 data creation pipeline completed successfully.")


main()
